In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_absolute_error, accuracy_score
from xgboost import XGBClassifier, XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# ── Load from SQLite ─────────────────────────────────────
conn = sqlite3.connect('digital_lending.db')
df = pd.read_sql_query("SELECT * FROM loans", conn)
conn.close()

print(f"Loaded: {df.shape}")
print(df.head(2))

Loaded: (765140, 27)
   loan_amount  interest_rate  term_months   income    dti  credit_score  \
0       5000.0          10.65         36.0  24000.0  27.65         739.0   
1       2500.0          15.27         60.0  30000.0   1.00         744.0   

   employment_length home_ownership lending_medium  default  ...  \
0               10.0           RENT            P2P        0  ...   
1                0.0           RENT            P2P        1  ...   

   income_segment  risk_interaction  digital_onboarding  \
0             Low          5.760177                   1   
1          Middle          0.083331                   0   

   upi_transaction_count  mobile_credit_score first_time_borrower urban_flag  \
0                     43           615.905387                   0          1   
1                     44           658.523390                   1          1   

  risk_tier  expected_loss  loan_id  
0    Medium       0.000000        1  
1      High    1367.052951        2  

[2 rows x 2

In [2]:
from sklearn.cluster import KMeans

# ── Features ─────────────────────────────────────────────
features = [
    'loan_amount', 'interest_rate', 'term_months', 'income',
    'dti', 'credit_score', 'employment_length',
    'loan_to_income', 'monthly_burden', 'high_dti_flag',
    'long_term_flag', 'cost_of_credit', 'risk_interaction',
    'digital_onboarding', 'upi_transaction_count',
    'mobile_credit_score', 'first_time_borrower', 'urban_flag',
    'home_ownership_enc', 'lending_medium_enc',
    'loan_size_enc', 'credit_tier_enc', 'income_segment_enc'
]

# ── Encode if not already ─────────────────────────────────
if 'home_ownership_enc' not in df.columns:
    df['home_ownership_enc'] = df['home_ownership'].astype('category').cat.codes
if 'lending_medium_enc' not in df.columns:
    df['lending_medium_enc'] = df['lending_medium'].astype('category').cat.codes
if 'loan_size_enc' not in df.columns:
    df['loan_size_enc'] = df['loan_size'].map({'Micro':0,'Small':1,'Medium':2,'Large':3}).fillna(0)
if 'credit_tier_enc' not in df.columns:
    df['credit_tier_enc'] = df['credit_tier'].map({'Poor':0,'Fair':1,'Good':2,'Very Good':3,'Exceptional':4,'Unknown':-1}).fillna(-1)
if 'income_segment_enc' not in df.columns:
    df['income_segment_enc'] = df['income_segment'].map({'Low':0,'Middle':1,'Upper Middle':2,'High':3,'Unknown':-1}).fillna(-1)

X = df[features].copy()
y_default = df['default']
y_risk = df['risk_tier'].map({'Low':0,'Medium':1,'High':2})
y_loss = df['expected_loss']

# ── Sample for speed (765k is large) ─────────────────────
sample_idx = df.sample(100000, random_state=42).index
X_sample = X.loc[sample_idx]
y_default_sample = y_default.loc[sample_idx]
y_risk_sample = y_risk.loc[sample_idx]
y_loss_sample = y_loss.loc[sample_idx]

# ── Scale ─────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

# ── Split ─────────────────────────────────────────────────
X_train, X_test, y_train_d, y_test_d = train_test_split(X_sample, y_default_sample, test_size=0.2, random_state=42)
_, _, y_train_r, y_test_r = train_test_split(X_sample, y_risk_sample, test_size=0.2, random_state=42)
_, _, y_train_l, y_test_l = train_test_split(X_sample, y_loss_sample, test_size=0.2, random_state=42)
X_train_s, X_test_s, _, _ = train_test_split(X_scaled, y_default_sample, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Default rate in sample: {y_default_sample.mean():.3f}")

Train: (80000, 23), Test: (20000, 23)
Default rate in sample: 0.040
